In [1]:
import os

docs_content = {
    "doc_01.txt": "Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.",
    "doc_02.txt": "Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto.",
    "doc_03.txt": "Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period.",
    "doc_04.txt": "Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue.",
    "doc_05.txt": "Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee.",
    "doc_06.txt": "If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items without requiring the customer to return the original item, unless the order value exceeds INR 1000, in which case a photo of the issue must be submitted through the report form before a replacement or refund is processed.",
    "doc_07.txt": "Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law.",
    "doc_08.txt": "Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered."
}

docs_dir = "docs"
os.makedirs(docs_dir, exist_ok=True)

for filename, content in docs_content.items():
    filepath = os.path.join(docs_dir, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Created {filepath}")

print("All 8 document files have been created in the 'docs' directory.")

Created docs/doc_01.txt
Created docs/doc_02.txt
Created docs/doc_03.txt
Created docs/doc_04.txt
Created docs/doc_05.txt
Created docs/doc_06.txt
Created docs/doc_07.txt
Created docs/doc_08.txt
All 8 document files have been created in the 'docs' directory.


In [3]:
!pip install -qU chromadb sentence-transformers

import os
import chromadb
from sentence_transformers import SentenceTransformer

# Initialize the embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Initialize ChromaDB client
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Create or get a collection
collection_name = "zepto_policies"
collection = chroma_client.get_or_create_collection(name=collection_name)

# Load documents, create embeddings, and add to ChromaDB
docs_dir = "docs"
doc_texts = []
doc_ids = []
metadatas = []

for i in range(1, 9):
    doc_id = f"doc_{i:02d}"
    filepath = os.path.join(docs_dir, f"{doc_id}.txt")
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    doc_texts.append(content)
    doc_ids.append(doc_id)
    metadatas.append({"source": filepath})

# Generate embeddings for the documents (each document is considered a chunk for now)
doc_embeddings = embedding_model.encode(doc_texts).tolist()

# Add documents to ChromaDB
collection.add(
    embeddings=doc_embeddings,
    documents=doc_texts,
    metadatas=metadatas,
    ids=doc_ids
)

print(f"Successfully loaded {len(doc_texts)} documents and stored their embeddings in ChromaDB collection '{collection_name}'.")
print(f"Total items in collection: {collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully loaded 8 documents and stored their embeddings in ChromaDB collection 'zepto_policies'.
Total items in collection: 8


In [4]:
PROMPT_TEMPLATE = """You are an AI assistant for Zepto's customer support. Your role is to provide accurate and concise answers to customer queries based *only* on the provided context. If the answer is not explicitly available in the context, state that you cannot answer the question. Do not make up information. Do not use outside knowledge.

Context:
{context}

Task: Answer the user's question based *only* on the provided context. Ensure your answer is factual, concise, and directly addresses the query. If the context does not contain the answer, respond with 'I cannot answer this question based on the provided Zepto policies.'.

Format: Provide a clear and direct answer.

Length: Keep the answer as short as possible while being comprehensive.

Examples:
User: How much does Zepto Pass+ cost and what benefits does it offer?
Assistant: Zepto Pass+ costs INR 99 per month. It offers free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members.

User: Can I return a damaged perishable item?
Assistant: Yes, grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect.

User: {query}
Assistant:"""

print("Prompt template has been defined.")

Prompt template has been defined.


In [5]:
import os
from typing import List, Literal, TypedDict
from pydantic import BaseModel

# Set MOCK_LLM for graded baseline (0 for real LLM if optional extension is pursued)
# os.environ["MOCK_LLM"] = "1" # Uncomment to explicitly set mock mode
# os.environ["MOCK_LLM"] = "0" # Uncomment and configure real LLM for optional extension

MOCK_LLM = os.getenv("MOCK_LLM", "1") == "1"
print(f"MOCK_LLM is set to: {MOCK_LLM} (True for mock mode, False for real LLM)")

class Answer(BaseModel):
    answer: str
    sources: List[str]
    confidence: float

print("Pydantic Answer model defined.")

# Define LangGraph State
class GraphState(TypedDict):
    query: str
    classification: Literal["policy_question", "general_question"]
    context: str # Retrieved context from ChromaDB
    answer: Answer # Structured answer including sources and confidence

print("LangGraph GraphState TypedDict defined.")

MOCK_LLM is set to: True (True for mock mode, False for real LLM)
Pydantic Answer model defined.
LangGraph GraphState TypedDict defined.


In [6]:
from langgraph.graph import StateGraph, END
import json

# --- Nodes for the LangGraph ---

def classify_intent(state: GraphState) -> GraphState:
    query = state["query"].lower()
    policy_keywords = [
        "delivery", "return", "refund", "membership",
        "tracking", "cancel", "gift card", "support hours"
    ]

    if MOCK_LLM: # Graded baseline: keyword heuristic
        if any(keyword in query for keyword in policy_keywords):
            classification = "policy_question"
        else:
            classification = "general_question"
    else:
        # Optional: Call LLM for classification (not implemented in this step)
        # For the purpose of this solution, we assume MOCK_LLM is True for graded tasks.
        # A real LLM call would involve a dedicated LLM chain for intent classification.
        classification = "general_question" # Placeholder if MOCK_LLM is False without actual LLM

    print(f"--- Classified intent: {classification} for query: '{state['query']}' ---")
    return {"classification": classification}


def retrieve_and_answer(state: GraphState) -> GraphState:
    query = state["query"]
    # Retrieve relevant documents from ChromaDB
    query_embedding = embedding_model.encode([query]).tolist()

    # Retrieve top 3 most similar chunks
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3,
        include=['documents', 'metadatas', 'distances']
    )

    retrieved_documents = results['documents'][0]
    retrieved_metadatas = results['metadatas'][0]
    retrieved_distances = results['distances'][0]

    context_snippets = []
    source_ids = []

    for i, doc in enumerate(retrieved_documents):
        context_snippets.append(doc) # Each document is a chunk for now
        source_ids.append(retrieved_metadatas[i]['source'].split('/')[-1].replace('.txt', ''))

    context = "\n\n".join(context_snippets)
    top_chunk_snippet = retrieved_documents[0][:200] + "..." if retrieved_documents else "No context found."

    final_answer: Answer
    if MOCK_LLM: # Graded baseline: canned templated answer
        answer_text = f"Based on the retrieved context: {top_chunk_snippet}"
        final_answer = Answer(
            answer=answer_text,
            sources=source_ids, # Sources are the retrieved document IDs
            confidence=1.0 # Fixed confidence for mock mode
        )
    else:
        # Optional: Call LLM with structured prompt template (not implemented in this step)
        # A real LLM call would format the PROMPT_TEMPLATE with context and query
        # and then parse the LLM's output into the Answer Pydantic model, with retry logic.
        # For the purpose of this solution, we assume MOCK_LLM is True for graded tasks.
        answer_text = "I cannot provide a real-LLM answer in this mock setup."
        final_answer = Answer(
            answer=answer_text,
            sources=[],
            confidence=0.0
        )

    print(f"--- Retrieved and Answered. Answer: {final_answer.answer[:50]}... ---")
    return {"context": context, "answer": final_answer}


def direct_answer(state: GraphState) -> GraphState:
    final_answer: Answer
    if MOCK_LLM: # Graded baseline: fixed canned string
        answer_text = "I can only answer questions about Zepto policies right now."
        final_answer = Answer(
            answer=answer_text,
            sources=[],
            confidence=1.0 # Fixed confidence for mock mode
        )
    else:
        # Optional: Call LLM directly (not implemented in this step)
        # A real LLM call would prompt the LLM directly without retrieval.
        # For the purpose of this solution, we assume MOCK_LLM is True for graded tasks.
        answer_text = "I cannot provide a real-LLM answer in this mock setup."
        final_answer = Answer(
            answer=answer_text,
            sources=[],
            confidence=0.0
        )
    print(f"--- Direct Answer: {final_answer.answer[:50]}... ---")
    return {"answer": final_answer}


# --- Build the LangGraph ---

workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("classify_intent", classify_intent)
workflow.add_node("retrieve_and_answer", retrieve_and_answer)
workflow.add_node("direct_answer", direct_answer)

# Set entry point
workflow.set_entry_point("classify_intent")

# Add conditional edge from classify_intent
workflow.add_conditional_edges(
    "classify_intent",
    lambda state: state["classification"],
    {
        "policy_question": "retrieve_and_answer",
        "general_question": "direct_answer",
    },
)

# Add edges to finish the graph
workflow.add_edge("retrieve_and_answer", END)
workflow.add_edge("direct_answer", END)

# Compile the graph
app = workflow.compile()

print("LangGraph workflow created and compiled successfully.")

LangGraph workflow created and compiled successfully.


In [8]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import nest_asyncio

# Define the request model for the /ask endpoint
class QueryRequest(BaseModel):
    query: str

# Initialize FastAPI app
app_fastapi = FastAPI(
    title="Zepto GenAI Service",
    description="A RAG-based GenAI service for Zepto's policy inquiries."
)

@app_fastapi.post("/ask", response_model=Answer)
async def ask_zepto_policy(request: QueryRequest):
    """Process a user query through the LangGraph and return a structured answer."""
    initial_state = {"query": request.query}

    # Invoke the LangGraph application
    final_state = app.invoke(initial_state)

    # The 'answer' in final_state is already a Pydantic Answer object
    return final_state["answer"]

print("FastAPI application and /ask endpoint defined.")

# To run FastAPI within a Colab notebook
# nest_asyncio is needed because Colab already runs an event loop
nest_asyncio.apply()

print("You can now run the FastAPI app using: !uvicorn main:app_fastapi --host 0.0.0.0 --port 8000 --reload")
print("or run it directly in a separate thread:")
print("import threading\nserver_thread = threading.Thread(target=uvicorn.run, args=(app_fastapi,), kwargs={\"host\": \"0.0.0.0\", \"port\": 8000})\nserver_thread.start()")

FastAPI application and /ask endpoint defined.
You can now run the FastAPI app using: !uvicorn main:app_fastapi --host 0.0.0.0 --port 8000 --reload
or run it directly in a separate thread:
import threading
server_thread = threading.Thread(target=uvicorn.run, args=(app_fastapi,), kwargs={"host": "0.0.0.0", "port": 8000})
server_thread.start()


In [9]:
import requests
import threading
import time

# Start the FastAPI server in a separate thread
# This allows the Colab notebook to continue executing while the server runs
server_thread = threading.Thread(target=uvicorn.run, args=(app_fastapi,), kwargs={"host": "0.0.0.0", "port": 8000})
server_thread.start()

print("FastAPI server started in a separate thread. Waiting for it to come online...")
time.sleep(5) # Give the server a few seconds to start up

base_url = "http://0.0.0.0:8000"

# --- Example 1: Policy Question (should trigger retrieval) ---
policy_query = "What are the delivery charges for Zepto?"
print(f"\n--- Testing policy question: '{policy_query}' ---")
response_policy = requests.post(f"{base_url}/ask", json={"query": policy_query})

print("Response Status Code:", response_policy.status_code)
print("Response JSON:", response_policy.json())

# --- Example 2: General Question (should trigger direct answer) ---
general_query = "What is the weather like in Mumbai today?"
print(f"\n--- Testing general question: '{general_query}' ---")
response_general = requests.post(f"{base_url}/ask", json={"query": general_query})

print("Response Status Code:", response_general.status_code)
print("Response JSON:", response_general.json())

# Optionally, stop the server thread if no longer needed
# This might require some more advanced signaling/shutdown logic for a clean exit.
# For simple demonstration, you can restart the kernel to stop the thread.
# print("\nServer demonstration complete. You may need to restart the kernel to fully stop the server thread.")

INFO:     Started server process [703]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


FastAPI server started in a separate thread. Waiting for it to come online...

--- Testing policy question: 'What are the delivery charges for Zepto?' ---
--- Classified intent: policy_question for query: 'What are the delivery charges for Zepto?' ---
--- Retrieved and Answered. Answer: Based on the retrieved context: Zepto delivers gro... ---
INFO:     127.0.0.1:58632 - "POST /ask HTTP/1.1" 200 OK
Response Status Code: 200
Response JSON: {'answer': "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del...", 'sources': ['doc_01', 'doc_03', 'doc_05'], 'confidence': 1.0}

--- Testing general question: 'What is the weather like in Mumbai today?' ---
--- Classified intent: general_question for query: 'What is the weather like in Mumbai today?' ---
--- Direct Answer: I can only answer questions about Zepto policies

## Dockerfile for FastAPI Application

In [10]:
%%writefile Dockerfile

# Use an official Python runtime as a parent image
FROM python:3.9-slim-buster

# Set the working directory in the container
WORKDIR /app

# Install system dependencies required by chromadb
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    gcc \
    && rm -rf /var/lib/apt/lists/*

# Copy the current directory contents into the container at /app
COPY . /app

# Install any needed packages specified in requirements.txt
# Assuming requirements.txt will contain: fastapi, uvicorn, nest_asyncio, langgraph, chromadb, sentence-transformers, pydantic, requests
# For simplicity, we'll install them directly here if no requirements.txt is provided yet.
RUN pip install --no-cache-dir \
    fastapi \
    uvicorn \
    nest_asyncio \
    langgraph \
    chromadb \
    sentence-transformers \
    pydantic \
    requests \
    "pydantic_core<2.13.0" # Pinning for compatibility with older pydantic versions if needed

# Expose the port that FastAPI will run on
EXPOSE 8000

# Command to run the uvicorn server
# Assuming the FastAPI app instance is named 'app_fastapi' in a file named 'main.py'
# For this notebook, the app is defined directly, so we need to create a main.py
CMD sh -c "python -c \"from main import app_fastapi; import uvicorn; uvicorn.run(app_fastapi, host='0.0.0.0', port=8000)\""


Writing Dockerfile


The `Dockerfile` has been created. However, to run the FastAPI app inside the Docker container, the application code (which is currently defined within the notebook) needs to be placed into a Python file, typically `main.py`. I will generate a `main.py` file containing the FastAPI application, the LangGraph setup, and all necessary imports and definitions.

In [11]:
%%writefile main.py

import os
from typing import List, Literal, TypedDict
from pydantic import BaseModel
from langgraph.graph import StateGraph, END
import chromadb
from sentence_transformers import SentenceTransformer
from fastapi import FastAPI
import uvicorn
import nest_asyncio

# --- Environment Configuration ---
MOCK_LLM = os.getenv("MOCK_LLM", "1") == "1"

# --- Pydantic Models ---
class Answer(BaseModel):
    answer: str
    sources: List[str]
    confidence: float

class QueryRequest(BaseModel):
    query: str

# --- LangGraph State Definition ---
class GraphState(TypedDict):
    query: str
    classification: Literal["policy_question", "general_question"]
    context: str # Retrieved context from ChromaDB
    answer: Answer # Structured answer including sources and confidence

# --- ChromaDB and Embedding Model Initialization ---
# These should ideally be initialized once globally or passed as dependencies
# For a simple app, we initialize them here.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="zepto_policies")

# --- Prompt Template (if needed for real LLM) ---
PROMPT_TEMPLATE = """You are an AI assistant for Zepto's customer support. Your role is to provide accurate and concise answers to customer queries based *only* on the provided context. If the answer is not explicitly available in the context, state that you cannot answer the question. Do not make up information. Do not use outside knowledge.\n\nContext:\n{context}\n\nTask: Answer the user's question based *only* on the provided context. Ensure your answer is factual, concise, and directly addresses the query. If the context does not contain the answer, respond with 'I cannot answer this question based on the provided Zepto policies.'.\n\nFormat: Provide a clear and direct answer.\n\nLength: Keep the answer as short as possible while being comprehensive.\n\nExamples:\nUser: How much does Zepto Pass+ cost and what benefits does it offer?\nAssistant: Zepto Pass+ costs INR 99 per month. It offers free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members.\n\nUser: Can I return a damaged perishable item?\nAssistant: Yes, grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect.\n\nUser: {query}\nAssistant:"""

# --- LangGraph Nodes ---
def classify_intent(state: GraphState) -> GraphState:
    query = state["query"].lower()
    policy_keywords = [
        "delivery", "return", "refund", "membership",
        "tracking", "cancel", "gift card", "support hours"
    ]

    if MOCK_LLM:
        if any(keyword in query for keyword in policy_keywords):
            classification = "policy_question"
        else:
            classification = "general_question"
    else:
        classification = "general_question" # Placeholder for real LLM

    print(f"[classify_intent] Classified: {classification} for '{state['query']}'")
    return {"classification": classification}

def retrieve_and_answer(state: GraphState) -> GraphState:
    query = state["query"]
    query_embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3,
        include=['documents', 'metadatas', 'distances']
    )

    retrieved_documents = results['documents'][0]
    retrieved_metadatas = results['metadatas'][0]

    context_snippets = []
    source_ids = []

    for i, doc in enumerate(retrieved_documents):
        context_snippets.append(doc)
        source_ids.append(retrieved_metadatas[i]['source'].split('/')[-1].replace('.txt', ''))

    context = "\n\n".join(context_snippets)
    top_chunk_snippet = retrieved_documents[0][:200] + "..." if retrieved_documents else "No context found."

    final_answer: Answer
    if MOCK_LLM:
        answer_text = f"Based on the retrieved context: {top_chunk_snippet}"
        final_answer = Answer(
            answer=answer_text,
            sources=source_ids,
            confidence=1.0
        )
    else:
        # Placeholder for real LLM integration with PROMPT_TEMPLATE and Pydantic retry
        answer_text = "I cannot provide a real-LLM answer in this mock setup."
        final_answer = Answer(
            answer=answer_text,
            sources=[],
            confidence=0.0
        )
    print(f"[retrieve_and_answer] Answer: {final_answer.answer[:50]}...")
    return {"context": context, "answer": final_answer}

def direct_answer(state: GraphState) -> GraphState:
    final_answer: Answer
    if MOCK_LLM:
        answer_text = "I can only answer questions about Zepto policies right now."
        final_answer = Answer(
            answer=answer_text,
            sources=[],
            confidence=1.0
        )
    else:
        # Placeholder for real LLM direct answer
        answer_text = "I cannot provide a real-LLM answer in this mock setup."
        final_answer = Answer(
            answer=answer_text,
            sources=[],
            confidence=0.0
        )
    print(f"[direct_answer] Answer: {final_answer.answer[:50]}...")
    return {"answer": final_answer}

# --- Build the LangGraph Workflow ---
workflow = StateGraph(GraphState)
workflow.add_node("classify_intent", classify_intent)
workflow.add_node("retrieve_and_answer", retrieve_and_answer)
workflow.add_node("direct_answer", direct_answer)
workflow.set_entry_point("classify_intent")
workflow.add_conditional_edges(
    "classify_intent",
    lambda state: state["classification"],
    {
        "policy_question": "retrieve_and_answer",
        "general_question": "direct_answer",
    },
)
workflow.add_edge("retrieve_and_answer", END)
workflow.add_edge("direct_answer", END)
app = workflow.compile()

# --- FastAPI Application ---
app_fastapi = FastAPI(
    title="Zepto GenAI Service",
    description="A RAG-based GenAI service for Zepto's policy inquiries."
)

@app_fastapi.post("/ask", response_model=Answer)
async def ask_zepto_policy(request: QueryRequest):
    initial_state = {"query": request.query}
    final_state = app.invoke(initial_state)
    return final_state["answer"]

# For local development within Colab (not strictly needed for Docker)
nest_asyncio.apply()

# To run this file directly with uvicorn:
# if __name__ == "__main__":
#     uvicorn.run(app_fastapi, host="0.0.0.0", port=8000)


Writing main.py


I've created `main.py` which contains all the necessary code to run the FastAPI application with the LangGraph logic. This file will be copied into the Docker image, and the Dockerfile is configured to run it using uvicorn.

## README.md for the Zepto GenAI Service

In [12]:
%%writefile README.md
# Zepto GenAI Service

This project implements a small but complete Generative AI (GenAI) service for Zepto, designed to answer policy-related customer inquiries. It leverages a RAG (Retrieval Augmented Generation) pipeline orchestrated by LangGraph, with a FastAPI application serving the API. The service is runnable locally and includes a Dockerfile for containerization.

## Architecture Description

The Zepto GenAI service follows a RAG architecture, divided into several key stages:

1.  **Ingestion & Embedding**: This stage involves processing the raw policy documents and converting them into numerical vector representations (embeddings). The `all-MiniLM-L6-v2` sentence transformer model is used for embedding, and these embeddings, along with the original document text, are stored in `ChromaDB` for efficient retrieval. Each policy document is treated as a single chunk.

2.  **FastAPI Application**: A `FastAPI` application exposes a `POST /ask` endpoint. This endpoint receives user queries, passes them to the LangGraph workflow, and returns a structured JSON response (`Answer` Pydantic model) containing the generated answer, source documents, and a confidence score.

3.  **LangGraph Orchestration**: The core logic of the RAG pipeline is orchestrated using `LangGraph`. It defines a `StateGraph` with a `TypedDict` (`GraphState`) to manage the conversational flow. The workflow consists of three main nodes and conditional edges:
    *   **`classify_intent`**: This node analyzes the incoming user query to determine its intent. It classifies queries as either `policy_question` or `general_question`. This node's behavior is gated by the `MOCK_LLM` environment variable.
    *   **`retrieve_and_answer`**: If the intent is `policy_question`, this node is activated. It takes the user's query, embeds it, and retrieves the top-k most relevant policy documents from `ChromaDB`. It then constructs an answer based on the retrieved context. This node's answer generation is also gated by `MOCK_LLM`.
    *   **`direct_answer`**: If the intent is `general_question`, this node is activated. It provides a canned, generic response, indicating that it can only answer policy-related questions. This node's behavior is also gated by `MOCK_LLM`.

    The `LangGraph` dynamically routes queries from `classify_intent` to either `retrieve_and_answer` or `direct_answer` based on the classification. Finally, both `retrieve_and_answer` and `direct_answer` nodes lead to the `END` state, returning the structured `Answer`.

4.  **MOCK_LLM Gating**: A critical aspect of this service is its support for a fully deterministic, rule-based mock mode. This is controlled by the `MOCK_LLM` environment variable:
    *   When `MOCK_LLM` is unset or `1` (default), the service operates in **mock mode**. `classify_intent` uses keyword heuristics, `retrieve_and_answer` returns a templated response based on retrieved snippets, and `direct_answer` returns a fixed string. No actual LLM calls are made, and no API keys are required.
    *   When `MOCK_LLM` is `0` (optional extension), the service is configured for **real LLM integration**. In this mode, `classify_intent` and `retrieve_and_answer` would ideally integrate with an external LLM (e.g., Groq) using the `PROMPT_TEMPLATE` for generation and Pydantic for structured output validation with retry logic. (Note: The real LLM paths are currently placeholders in the provided code).

### Data Flow

User Query --> FastAPI (`/ask` endpoint) --> LangGraph (`classify_intent`) --("policy_question")--> LangGraph (`retrieve_and_answer`) --> ChromaDB (retrieval) --> LangGraph (`retrieve_and_answer` generates answer) --> FastAPI (returns structured JSON)

User Query --> FastAPI (`/ask` endpoint) --> LangGraph (`classify_intent`) --("general_question")--> LangGraph (`direct_answer` generates answer) --> FastAPI (returns structured JSON)

## Local Development Setup

### Prerequisites

*   Python 3.9+
*   Docker (for containerized deployment)
*   `pip` for package installation

### Steps to Run

1.  **Clone the repository (or set up the files from the notebook):** Ensure you have `main.py`, `Dockerfile`, and the `docs` directory (containing `doc_01.txt` to `doc_08.txt`) in your working directory.

2.  **Build the Docker image:**

    ```bash
    docker build -t zepto-genai-service .
    ```

3.  **Run the Docker container:**

    The service will be accessible on `http://localhost:8000`. By default, it runs in `MOCK_LLM=1` mode.

    ```bash
    docker run -p 8000:8000 -e MOCK_LLM=1 zepto-genai-service
    ```

    *(Optional: To simulate the real LLM path (which is a placeholder), you could run with `-e MOCK_LLM=0`, but no actual LLM integration is active in the provided code.)*

4.  **Make API requests:**

    You can use `curl` or any HTTP client to interact with the service.

### Example API Calls (Mock Mode `MOCK_LLM=1`)

#### Example 1: Policy Question

**Query:** `What are the delivery charges for Zepto?`

**Request:**

```bash
curl -X POST "http://localhost:8000/ask" \
     -H "Content-Type: application/json" \
     -d '{"query": "What are the delivery charges for Zepto?"}'
```

**Response:**

```json
{
  "answer": "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del...",
  "sources": [
    "doc_01",
    "doc_03",
    "doc_05"
  ],
  "confidence": 1.0
}
```

#### Example 2: General Question

**Query:** `What is the weather like in Mumbai today?`

**Request:**

```bash
curl -X POST "http://localhost:8000/ask" \
     -H "Content-Type: application/json" \
     -d '{"query": "What is the weather like in Mumbai today?"}'
```

**Response:**

```json
{
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}
```

## Further Extensions (Ungraded)

*   **Real LLM Integration**: Implement actual LLM calls (e.g., using Groq, OpenAI, or other providers) in `classify_intent` and `retrieve_and_answer` when `MOCK_LLM=0`.
*   **Live Cloud Deployment**: Deploy the FastAPI application to a cloud platform (e.g., Hugging Face Spaces, Google Cloud Run, AWS Fargate).
*   **Advanced RAG**: Implement chunking strategies, more sophisticated retrieval, or re-ranking for better context quality.
*   **Error Handling**: Add more robust error handling and logging.


Writing README.md


The `README.md` file has been generated with the architecture description and example API calls. This completes all the required tasks for the graded baseline. If you would like to proceed with any optional extensions, please let me know!

## `requirements.txt`

In [13]:
%%writefile requirements.txt
fastapi
uvicorn
nest_asyncio
langgraph
chromadb
sentence-transformers
pydantic
requests
pydantic_core<2.13.0


Writing requirements.txt
